# Hybrid Biomedical Image Analysis — Fluorescence Microscopy Nuclei

**Pipeline:** raw image → segmentation → quantitative region features → structured JSON record → narrative.

Covers Tasks 1–4 of the assignment plus two extensions (loss ablation, robustness trace).

---

### How to run

1. Put `nuclei_dataset/` next to this notebook (or set `DATA_ROOT` below).
2. Install: `pip install torch torchvision scikit-image matplotlib pandas seaborn`
3. **For the LLM steps**, start Ollama locally and pull the models:
   ```bash
   ollama serve
   ollama pull llama3.2-vision
   ollama pull llama3.1:8b
   ```
   If Ollama is not reachable the notebook still runs end to end: the LLM steps fall back to a
   deterministic rule-based stub and every record is tagged `llm_source = "offline-stub"`.
   **Re-run with Ollama running before submitting** so the reported narratives are real model output.
4. Run all cells. Figures land in `figs/`, numeric results in `out/`.

> **Educational use only.** None of these models are cleared for clinical use. The LLM steps are
> descriptive, never diagnostic, and every number in a record is computed in Python, not generated.

In [ ]:
# ---- Ollama on Colab (fixed) -----------------------------------------
!sudo apt-get -qq update && sudo apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import shutil
assert shutil.which('ollama'), 'install still failed — read the output above'
print('binary at:', shutil.which('ollama'))

In [ ]:
import subprocess, os, time, urllib.request
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['ollama', 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        print('ollama serve is up'); break
    except Exception:
        time.sleep(2)
else:
    print('server did not start')

In [ ]:
# Pull the language models. llama3.2-vision is the model named in the brief;
# llava:7b is a smaller fallback used only if the first one cannot serve requests.
!ollama pull llama3.2-vision
!ollama pull llava:7b
!ollama pull llama3.1:8b
!ollama list

In [ ]:
import json, os, re, base64, time, urllib.request, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings('ignore')

DATA_ROOT = Path('nuclei_dataset')       # <-- edit if your dataset lives elsewhere
FIGS = Path('figs'); FIGS.mkdir(exist_ok=True)
OUT  = Path('out');  OUT.mkdir(exist_ok=True)

IMG_SIZE = 256          # common size all images are resized to
CROP     = 128          # random crop used for U-Net training
SEED     = 0
np.random.seed(SEED)

# ---- locate the dataset -------------------------------------------------
# Handles all three cases: an unzipped folder, a .zip sitting next to the
# notebook (e.g. uploaded to Colab), or nothing at all.
import zipfile

def find_dataset(start=DATA_ROOT):
    if start.is_dir() and (start/'metadata.csv').exists():
        return start
    # a zip anywhere in the working directory? unpack it
    for z in list(Path('.').glob('*.zip')) + list(Path('/content').glob('*.zip') if Path('/content').exists() else []):
        try:
            with zipfile.ZipFile(z) as f:
                f.extractall(z.parent)
            print('unpacked', z.name)
        except zipfile.BadZipFile:
            pass
    hits = sorted(Path('.').rglob('metadata.csv'))
    if not hits and Path('/content').exists():
        hits = sorted(Path('/content').rglob('metadata.csv'))
    if hits:
        return hits[0].parent
    # last resort: clone from the module repository
    print('dataset not found locally - cloning from GitHub...')
    os.system('git clone --depth 1 https://github.com/Nickolay-K/Assingnment-3-dataset dataset_repo')
    hits = sorted(Path('.').rglob('metadata.csv'))
    return hits[0].parent if hits else start

DATA_ROOT = find_dataset()
assert DATA_ROOT.is_dir() and (DATA_ROOT/'metadata.csv').exists(), (
    f'dataset not found (looked at {DATA_ROOT}). Unzip nuclei_dataset.zip next '
    'to this notebook, or set DATA_ROOT to the folder containing metadata.csv.')
print('DATA_ROOT =', DATA_ROOT.resolve())

for sub in ['train/images', 'train/masks', 'val/images', 'val/masks',
            'test/images', 'test/masks', 'test_corrupted/images']:
    n = len(list((DATA_ROOT/sub).glob('*.png'))) if (DATA_ROOT/sub).exists() else 0
    print(f'{sub:26s} {n:4d} png' + ('' if n else '   <-- MISSING'))

---
## Task 1 — Data preparation, EDA and multimodal LLM description

### 1.1 Loading, grayscale conversion and resizing

The raw images are 256×256 RGB with a DAPI-like blue stain on a dark field. We convert to
grayscale with PIL's ITU-R 601-2 luma transform (`L = 0.299R + 0.587G + 0.114B`) and resize to a
common 256×256. Note the trade-off recorded in the report: because the signal sits almost entirely
in the blue channel, luma conversion scales it by ~0.114 and compresses the dynamic range. It is
kept because the assignment specifies grayscale, and Otsu is invariant to the monotone rescaling.

In [ ]:
def load_gray(path, size=IMG_SIZE):
    """RGB PNG -> grayscale, resized, float32 in [0,1]."""
    im = Image.open(path).convert('L')
    if im.size != (size, size):
        im = im.resize((size, size), Image.BILINEAR)
    return np.asarray(im, dtype=np.float32) / 255.0

def load_mask(path, size=IMG_SIZE):
    im = Image.open(path).convert('L')
    if im.size != (size, size):
        im = im.resize((size, size), Image.NEAREST)
    return (np.asarray(im) > 127).astype(np.float32)

def split_ids(split):
    return sorted(p.stem for p in (DATA_ROOT/split/'images').glob('*.png'))

def load_split(split, size=IMG_SIZE):
    ids = split_ids(split)
    X = np.stack([load_gray(DATA_ROOT/split/'images'/f'{i}.png', size) for i in ids])
    Y = np.stack([load_mask(DATA_ROOT/split/'masks'/f'{i}.png', size) for i in ids])
    return ids, X, Y

meta = pd.read_csv(DATA_ROOT/'metadata.csv')
meta.columns = [c.strip() for c in meta.columns]
print(meta.groupby(['split','density']).size())
print('\nobjects per image:', meta.n_objects.min(), '-', meta.n_objects.max(),
      '| mean', round(meta.n_objects.mean(),1))

### 1.2 Exploratory data analysis

In [ ]:
reps = {d: meta[(meta.split=='train') & (meta.density==d)].iloc[0].image_id
        for d in ['sparse','normal','dense','clustered']}

fig, axes = plt.subplots(2, 4, figsize=(11, 5.6))
for j,(dens,iid) in enumerate(reps.items()):
    g = load_gray(DATA_ROOT/'train'/'images'/f'{iid}.png')
    m = load_mask(DATA_ROOT/'train'/'masks'/f'{iid}.png')
    n = int(meta.loc[meta.image_id==iid,'n_objects'].iloc[0])
    axes[0,j].imshow(g, cmap='gray', vmin=0, vmax=float(np.percentile(g,99.9)))
    axes[0,j].set_title(f'{dens} — {iid}\nn={n} nuclei', fontsize=9)
    axes[1,j].imshow(m, cmap='gray'); axes[1,j].set_title('ground-truth mask', fontsize=9)
    axes[0,j].axis('off'); axes[1,j].axis('off')
plt.tight_layout(); plt.savefig(FIGS/'fig1_samples.png', dpi=160, bbox_inches='tight'); plt.show()

In [ ]:
ids = split_ids('train')
allpix = np.concatenate([load_gray(DATA_ROOT/'train'/'images'/f'{i}.png').ravel() for i in ids])
fg, bg = [], []
for i in ids[:40]:
    g = load_gray(DATA_ROOT/'train'/'images'/f'{i}.png')
    m = load_mask(DATA_ROOT/'train'/'masks'/f'{i}.png').astype(bool)
    fg.append(g[m]); bg.append(g[~m])
fg, bg = np.concatenate(fg), np.concatenate(bg)

fig, ax = plt.subplots(1,3, figsize=(12,3.4))
ax[0].hist(allpix, bins=64, color='#3b6ea5'); ax[0].set_yscale('log')
ax[0].set_title('All train pixels'); ax[0].set_xlabel('intensity'); ax[0].set_ylabel('count (log)')
ax[1].hist(bg, bins=64, alpha=.65, label='background', color='#888', density=True)
ax[1].hist(fg, bins=64, alpha=.65, label='nucleus', color='#c1443c', density=True)
ax[1].set_yscale('log'); ax[1].legend(fontsize=8)
ax[1].set_title('Foreground vs background (GT mask)'); ax[1].set_xlabel('intensity')
for d,c in zip(['sparse','normal','dense','clustered'], ['#4c9f70','#3b6ea5','#d18b2c','#8e5ea2']):
    s = meta[(meta.split=='train') & (meta.density==d)]
    ax[2].scatter(s.n_objects, s.area_fraction, s=22, label=d, color=c)
ax[2].set_xlabel('n_objects (GT)'); ax[2].set_ylabel('foreground area fraction')
ax[2].set_title('Density regimes'); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIGS/'fig2_eda_hist.png', dpi=160, bbox_inches='tight'); plt.show()

print(f'foreground pixel fraction: {len(fg)/(len(fg)+len(bg)):.4f}')
print(f'mean intensity  fg={fg.mean():.4f}  bg={bg.mean():.4f}')

The histogram is strongly bimodal with a wide empty valley, and only ~8% of pixels are foreground.
Two consequences drive the rest of the report: (a) global Otsu thresholding should already be close
to optimal on clean images, so the U-Net has very little headroom; (b) the class imbalance is ~11:1,
which is why a plain BCE loss is a poor choice and why the loss ablation below matters.

### 1.3 Local multimodal LLM (llama3.2-vision via Ollama)

The prompt is engineered around four constraints:

| Constraint | Why |
|---|---|
| Anchored as **descriptive, not diagnostic** | a VLM asked an open medical question will volunteer clinical language it cannot support |
| **Closed vocabularies** for every categorical field | turns free-text hallucination into a validation failure that can be detected automatically |
| **`"uncertain"` explicitly permitted** | without an escape hatch the model is forced to commit, and it commits confidently |
| **"Do not count objects"** | counting is exactly what a VLM is worst at and what the classical pipeline is best at |

In [ ]:
OLLAMA_URL = 'http://localhost:11434/api/generate'
TEXT_MODEL = 'llama3.1:8b'
VISION_CANDIDATES = ['llama3.2-vision', 'llava:7b']   # tried in order
VISION_MODEL = None                                   # chosen by the probe below

def ollama_available():
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3); return True
    except Exception:
        return False

def call_llm(prompt, image_path=None, model=None, temperature=0.2, seed=None,
             timeout=1200, quiet=False):
    """Call a local Ollama model. Returns (text, source): 'ollama' or 'offline-stub'."""
    model = model or (VISION_MODEL if image_path else TEXT_MODEL)
    payload = {'model': model, 'prompt': prompt, 'stream': False, 'keep_alive': '10m',
               'options': {'temperature': temperature, 'num_predict': 500}}
    if seed is not None:
        payload['options']['seed'] = seed
    if image_path is not None:
        payload['images'] = [base64.b64encode(Path(image_path).read_bytes()).decode()]
    try:
        req = urllib.request.Request(OLLAMA_URL, data=json.dumps(payload).encode(),
                                     headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read().decode())['response'], 'ollama'
    except urllib.error.HTTPError as e:
        if not quiet:
            print(f'  [call_llm HTTP {e.code}] {e.read().decode()[:400]}')
        return None, 'offline-stub'
    except Exception as e:
        if not quiet:
            print(f'  [call_llm FAILED] {type(e).__name__}: {e}')
        return None, 'offline-stub'

def split_narrative(text):
    """Prose with every JSON block removed - works whether the JSON comes first or last."""
    if not text:
        return ''
    t = re.sub(r'```(?:json)?', '', text)
    keep, depth = [], 0
    for ch in t:
        if ch == '{':
            depth += 1
        if depth == 0:
            keep.append(ch)
        if ch == '}':
            depth = max(0, depth - 1)
    prose = re.sub(r'PART\s*[12]\s*[:\-]?', ' ', ''.join(keep))
    return re.sub(r'\s+', ' ', prose).strip()

import urllib.error
USE_OLLAMA = ollama_available()
print('Ollama reachable:', USE_OLLAMA)

# ---- pick a vision model that actually serves an image request ----------
if USE_OLLAMA:
    probe = Path('/tmp/probe.png')
    Image.fromarray((np.random.rand(64, 64) * 255).astype('uint8')).save(probe)
    for cand in VISION_CANDIDATES:
        txt, _ = call_llm('Reply with the single word: ok.', image_path=probe,
                          model=cand, timeout=900, quiet=True)
        if txt is not None:
            VISION_MODEL = cand
            print(f'vision model in use: {VISION_MODEL}')
            break
        print(f'  {cand}: cannot serve image requests here, trying next')
    if VISION_MODEL is None:
        print('NO VISION MODEL AVAILABLE - Task 1 will be skipped and must be '
              'reported as a limitation.')


In [ ]:
PROMPT_T1_NAIVE = 'What do you see in this medical image?'

PROMPT_T1_STRUCTURED = '''You are an image-description assistant for a microscopy \
teaching dataset. You describe only what is visually present. You are NOT a \
diagnostic tool: never name a disease, never state or imply a clinical finding, \
never comment on patient status.

Describe the attached image and return ONE JSON object and nothing else - no \
preamble, no markdown fences, no commentary after it.

Schema (all keys required, values must come from the listed vocabularies):
{
  "modality": one of ["fluorescence_microscopy","brightfield_microscopy","histology",
                      "radiograph","ct","mri","ultrasound","uncertain"],
  "tissue_type": one of ["cell_nuclei","cell_culture","tissue_section","other","uncertain"],
  "notable_features": array of 2-5 short descriptive strings, each purely visual
                      (e.g. "bright rounded objects on dark background",
                       "objects vary in size", "some objects touch"),
  "image_quality": one of ["good","adequate","poor","uncertain"],
  "confidence": one of ["high","medium","low"]
}

Rules:
- If a property cannot be determined from the pixels alone, use "uncertain" rather than \
guessing. "uncertain" is always an acceptable answer.
- Do not count objects; you are not able to count reliably. Do not give measurements.
- Do not mention anything that is not visible in this specific image.
'''

print(PROMPT_T1_STRUCTURED)

In [ ]:
REP_IMAGE = DATA_ROOT/'train'/'images'/f"{reps['normal']}.png"
print('representative image:', REP_IMAGE, '| model:', VISION_MODEL)

t1 = {}
if VISION_MODEL:
    t1['naive'], _      = call_llm(PROMPT_T1_NAIVE, image_path=REP_IMAGE)
    t1['structured'], _ = call_llm(PROMPT_T1_STRUCTURED, image_path=REP_IMAGE)
    print('--- NAIVE PROMPT ---\n', t1['naive'])
    print('\n--- STRUCTURED PROMPT ---\n', t1['structured'])
else:
    print('skipped - no working vision model')

### 1.4 Repeated runs are not identical

Three calls with the same prompt, same image and the same temperature. Sampling makes the output
non-deterministic, which is the core argument for putting a schema between the model and the record.

In [ ]:
t1_repeats = []
if VISION_MODEL:
    for k in range(3):
        txt, _ = call_llm(PROMPT_T1_STRUCTURED, image_path=REP_IMAGE, temperature=0.2)
        t1_repeats.append(txt)
        print(f'--- run {k+1} ---\n{txt}\n')
    same = len(set(t1_repeats)) == 1
    print('all three byte-identical:', same)
    (OUT/'task1_vlm.json').write_text(json.dumps(
        {'vision_model': VISION_MODEL, 'naive': t1.get('naive'),
         'structured': t1.get('structured'), 'repeats': t1_repeats,
         'identical': bool(same)}, indent=2))
else:
    print('skipped - no working vision model')

A useful extra check: extract the JSON from each repeat and compare **fields**, not strings. A good
prompt makes the *record* stable even when the *wording* is not — that stability is what the
downstream CSV depends on.

In [ ]:
def extract_json(text):
    """Pull the last complete JSON object out of a model reply."""
    if not text: return None
    text = re.sub(r'```(?:json)?', '', text)
    best = None
    for m in re.finditer(r'\{', text):
        depth = 0
        for i in range(m.start(), len(text)):
            if text[i] == '{': depth += 1
            elif text[i] == '}':
                depth -= 1
                if depth == 0:
                    try: best = json.loads(text[m.start():i+1])
                    except json.JSONDecodeError: pass
                    break
    return best

if t1_repeats:
    recs = [extract_json(t) for t in t1_repeats]
    print(pd.DataFrame(recs).to_string())

---
## Task 2 — Classical features and numbers-first LLM interpretation

### 2.1 Otsu + morphological cleanup + connected components

In [ ]:
from skimage import measure, morphology, segmentation
from skimage.filters import threshold_otsu
from skimage.color import label2rgb
from skimage.feature import peak_local_max
from skimage.segmentation import watershed as ws_split
from scipy import ndimage as ndi

def otsu_segment(gray, min_size=30, closing_radius=2):
    thr = threshold_otsu(gray)
    b = gray > thr
    b = morphology.binary_closing(b, morphology.disk(closing_radius))   # bridge 1-px gaps
    b = morphology.remove_small_holes(b, area_threshold=64)             # fill nucleoli holes
    b = morphology.remove_small_objects(b, min_size=min_size)           # drop noise specks
    return b, thr

def label_objects(binary, watershed=True, min_distance=4):
    """Label components; optionally split touching nuclei with a distance-transform watershed."""
    if not watershed:
        return measure.label(binary)
    dist = ndi.distance_transform_edt(binary)
    coords = peak_local_max(dist, min_distance=min_distance, labels=binary)
    markers = np.zeros(dist.shape, dtype=int)
    for k,(r,c) in enumerate(coords, start=1): markers[r,c] = k
    markers = ndi.label(markers > 0)[0]
    return ws_split(-dist, markers, mask=binary)

In [ ]:
PROPS = ('label','area','perimeter','eccentricity','solidity','extent',
         'equivalent_diameter','major_axis_length','minor_axis_length',
         'orientation','mean_intensity','max_intensity','min_intensity','centroid')

def region_features(labels, gray):
    if labels.max() == 0:
        return pd.DataFrame(columns=list(PROPS))
    df = pd.DataFrame(measure.regionprops_table(labels, intensity_image=gray, properties=PROPS))
    df['circularity']  = np.where(df.perimeter>0, 4*np.pi*df.area/df.perimeter**2, np.nan)
    df['aspect_ratio'] = np.where(df.minor_axis_length>0,
                                  df.major_axis_length/df.minor_axis_length, np.nan)
    return df

def border_count(df, image_size=IMG_SIZE, margin=2):
    if len(df)==0: return 0
    r, c, rad = df['centroid-0'], df['centroid-1'], df.equivalent_diameter/2
    return int(((r-rad<margin)|(c-rad<margin)|(r+rad>image_size-margin)|(c+rad>image_size-margin)).sum())

In [ ]:
IID  = reps['normal']
gray = load_gray(DATA_ROOT/'train'/'images'/f'{IID}.png')
binary, thr = otsu_segment(gray)
lab_cc = label_objects(binary, watershed=False)
lab_ws = label_objects(binary, watershed=True)
feat   = region_features(lab_ws, gray)

fig, ax = plt.subplots(1,5, figsize=(15,3.2))
ax[0].imshow(gray, cmap='gray', vmin=0, vmax=np.percentile(gray,99.9)); ax[0].set_title(f'grayscale {IID}', fontsize=9)
ax[1].hist(gray.ravel(), bins=64, color='#3b6ea5'); ax[1].axvline(thr, color='r', lw=1.4)
ax[1].set_yscale('log'); ax[1].set_title(f'histogram, Otsu t={thr:.3f}', fontsize=9)
ax[2].imshow(binary, cmap='gray'); ax[2].set_title('Otsu + morphology', fontsize=9)
ax[3].imshow(label2rgb(lab_cc, bg_label=0)); ax[3].set_title(f'connected comps (n={lab_cc.max()})', fontsize=9)
ax[4].imshow(label2rgb(lab_ws, bg_label=0)); ax[4].set_title(f'watershed split (n={lab_ws.max()})', fontsize=9)
for a in ax:
    if a is not ax[1]: a.axis('off')
plt.tight_layout(); plt.savefig(FIGS/'fig3_classical.png', dpi=160, bbox_inches='tight'); plt.show()

feat.round(2).to_csv(OUT/f'features_{IID}.csv', index=False)
display(feat[['label','area','eccentricity','solidity','circularity','mean_intensity']].head(8).round(3))
print('true n_objects:', int(meta.loc[meta.image_id==IID,'n_objects'].iloc[0]))

### 2.2 Quality metrics and the measurement dictionary

Two cheap image-level statistics act as the pipeline's own quality gate. They are computed from the
pixels, not from the model, so they cannot be talked out of firing.

* `contrast_score` = mean(pixels above Otsu) − mean(pixels below) — collapses under contrast loss.
* `focus_score` = var(Laplacian) / var(image) — collapses under blur.

In [ ]:
def quality_metrics(gray):
    thr = threshold_otsu(gray)
    fg, bg = gray[gray>thr], gray[gray<=thr]
    contrast = float(fg.mean()-bg.mean()) if fg.size else 0.0
    focus = float(ndi.laplace(gray.astype(np.float64)).var() / (gray.var()+1e-9))
    return dict(otsu_threshold=round(float(thr),4),
                contrast_score=round(contrast,4), focus_score=round(focus,4))

def measure_field(image_id, gray, mask):
    """Segmentation mask + grayscale -> the measurement dict handed to the LLM."""
    labels = label_objects(mask.astype(bool), watershed=True)
    df = region_features(labels, gray)
    m = dict(image_id=image_id, image_size=f'{IMG_SIZE}x{IMG_SIZE}', n_objects=int(len(df)),
             area_fraction=round(float(mask.sum())/mask.size,4), **quality_metrics(gray))
    zero = ['mean_area','median_area','min_area','max_area','std_area','mean_equiv_diameter',
            'mean_eccentricity','mean_solidity','mean_circularity','mean_aspect_ratio','mean_intensity']
    if len(df)==0:
        m.update({k:0.0 for k in zero}); m['n_border_objects']=0; return m, df
    m.update(dict(
        mean_area=round(float(df.area.mean()),1), median_area=round(float(df.area.median()),1),
        min_area=round(float(df.area.min()),1),  max_area=round(float(df.area.max()),1),
        std_area=round(float(df.area.std(ddof=0)),1),
        mean_equiv_diameter=round(float(df.equivalent_diameter.mean()),2),
        mean_eccentricity=round(float(df.eccentricity.mean()),3),
        mean_solidity=round(float(df.solidity.mean()),3),
        mean_circularity=round(float(df.circularity.mean()),3),
        mean_aspect_ratio=round(float(df.aspect_ratio.mean()),2),
        mean_intensity=round(float(df.mean_intensity.mean()),3),
        n_border_objects=border_count(df)))
    return m, df

m_otsu, _ = measure_field(IID, gray, binary)
print(json.dumps(m_otsu, indent=2))

### 2.3 The numbers-first LLM step

The model receives **only** the measurement dictionary — never the image. Two safeguards sit around it:

1. **Numbers are injected, never generated.** `n_objects`, `mean_area` and `image_id` are copied from
   the measurement dict into the record after the call.
2. **A validator holds the model to the schema.** Out-of-vocabulary labels become `"uncertain"`,
   contradicted numbers are overwritten, extra keys are dropped, and every substitution is logged in
   a `corrections` list so a reviewer can see exactly where the model disagreed with the measurements.

In [ ]:
PROMPT_T2 = '''You are a scientific writing assistant. You will be given ONLY a set \
of measurements that were computed by a classical image-processing pipeline \
(Otsu threshold, morphological cleanup, connected-component labelling, \
scikit-image regionprops). You have NOT seen the image and you must not pretend \
to have seen it.

Write your answer as exactly two parts:

PART 1 - one paragraph (3-5 sentences) in plain English that restates and \
interprets the measurements. Use only numbers that appear in the measurements \
below. Do not invent any value. Do not give a diagnosis or any clinical opinion. \
Describe the objects as "segmented objects", not as a biological or clinical entity \
beyond what the measurements state.

PART 2 - one JSON object on its own, and nothing after it:
{
  "n_objects": integer, copied exactly from the measurements,
  "density_class": one of ["sparse","moderate","dense","uncertain"],
  "shape_regularity": one of ["regular","mixed","irregular","uncertain"],
  "quality_flag": one of ["ok","low_contrast","blurred","segmentation_unreliable","uncertain"]
}

Use "uncertain" for any field the measurements do not support. Do not add keys.

MEASUREMENTS:
{measurements}
'''

PROMPT_T4 = '''You are the reporting step of an auditable image-analysis pipeline. \
You are given measurements computed from a U-Net segmentation mask by \
scikit-image regionprops. You have not seen the image.

Return exactly two parts.

PART 1 - one JSON object, nothing else on those lines:
{
  "image_id": string, copied exactly,
  "n_objects": integer, copied exactly,
  "mean_area": number, copied exactly,
  "density_class": one of ["sparse","moderate","dense","uncertain"],
  "quality_flag": one of ["ok","low_contrast","blurred","segmentation_unreliable","uncertain"]
}

PART 2 - one paragraph (3-4 sentences) describing the field, aimed at a technician \
reviewing the run. State the object count, the typical object size and the spread, \
the shape summary, and any quality concern. Use only the numbers given. Do not \
diagnose, do not speculate about the sample's origin, and do not describe anything \
that is not in the measurements. If a quality flag other than "ok" is set, say \
explicitly that the downstream numbers should be treated as provisional.

MEASUREMENTS:
{measurements}
'''

In [ ]:
VOCAB = {
  'density_class':    ['sparse','moderate','dense','uncertain'],
  'shape_regularity': ['regular','mixed','irregular','uncertain'],
  'quality_flag':     ['ok','low_contrast','blurred','segmentation_unreliable','uncertain'],
}

def classify(m):
    """Rule-based reference labels: the offline stub AND the yardstick the validator uses.
    Thresholds calibrated on the clean training split (contrast_score mean 0.241, min 0.198;
    focus_score min 0.28)."""
    n, af = m['n_objects'], m['area_fraction']
    density = ('sparse'   if (n < 15 and af < 0.05) else
               'dense'    if (n >= 35 or af > 0.12) else 'moderate')
    sol, circ = m.get('mean_solidity',1.0), m.get('mean_circularity',1.0)
    shape = ('regular'   if sol > 0.95 and circ > 0.85 else
             'irregular' if sol < 0.88 or circ < 0.70 else 'mixed')
    if   m.get('focus_score',1)    < 0.15: q = 'blurred'
    elif m.get('contrast_score',1) < 0.15: q = 'low_contrast'
    elif n == 0 or sol < 0.80:             q = 'segmentation_unreliable'
    else:                                  q = 'ok'
    return density, shape, q

def validate(record, computed, schema_keys):
    """Coerce a model record onto the schema and the measured numbers. -> (record, corrections)"""
    rec, fixes, record = {}, [], (record or {})
    for k in schema_keys:
        v = record.get(k)
        if k in computed:
            if v is None:
                fixes.append(f'{k}: missing -> computed value')
            elif isinstance(computed[k], (int,float)) and not isinstance(computed[k], bool):
                try:
                    if abs(float(v)-float(computed[k])) > 1e-6:
                        fixes.append(f'{k}: model said {v} -> computed {computed[k]}')
                except (TypeError, ValueError):
                    fixes.append(f'{k}: non-numeric {v!r} -> computed {computed[k]}')
            elif str(v) != str(computed[k]):
                fixes.append(f'{k}: model said {v!r} -> computed {computed[k]!r}')
            rec[k] = computed[k]
        elif k in VOCAB:
            if isinstance(v,str) and v.lower() in VOCAB[k]: rec[k] = v.lower()
            else:
                fixes.append(f'{k}: {v!r} not in vocabulary -> uncertain'); rec[k] = 'uncertain'
        else:
            rec[k] = v if isinstance(v,list) else ([] if v is None else [str(v)])
    extra = [k for k in record if k not in schema_keys]
    if extra: fixes.append(f'dropped unexpected keys: {extra}')
    return rec, fixes

In [ ]:
def stub_reply(m, kind):
    """Deterministic offline fallback so the pipeline runs without Ollama."""
    density, shape, quality = classify(m)
    if m['n_objects'] == 0:
        para = ('The segmentation returned no objects, so no per-object statistics could be '
                'computed; the field should be re-examined before the measurements are used.')
    else:
        warn = '' if quality=='ok' else (f" The quality flag '{quality}' was raised, so these "
               'numbers should be treated as provisional and reviewed before use.')
        lead = f"Image {m['image_id']}: " if kind=='t4' else ''
        para = (f"{lead}The segmentation isolated {m['n_objects']} objects covering "
                f"{100*m['area_fraction']:.1f}% of the {m['image_size']} field, which corresponds to a "
                f"{density} field. Objects have a mean area of {m['mean_area']:.0f} pixels "
                f"(median {m['median_area']:.0f}, range {m['min_area']:.0f}-{m['max_area']:.0f}), "
                f"equivalent to a typical diameter of about {m['mean_equiv_diameter']:.0f} pixels. "
                f"Shape statistics are {shape}: mean solidity {m['mean_solidity']:.2f}, mean "
                f"circularity {m['mean_circularity']:.2f} and mean eccentricity "
                f"{m['mean_eccentricity']:.2f}. Mean intensity inside the objects is "
                f"{m['mean_intensity']:.2f} against a background separation of "
                f"{m['contrast_score']:.2f}.{warn}")
    if kind=='t2':
        rec = {'n_objects':int(m['n_objects']), 'density_class':density,
               'shape_regularity':shape, 'quality_flag':quality}
    else:
        rec = {'image_id':m['image_id'], 'n_objects':int(m['n_objects']),
               'mean_area':round(float(m['mean_area']),1),
               'density_class':density, 'quality_flag':quality}
    return para + '\n\n' + json.dumps(rec)

def run_step(prompt_template, m, kind, temperature=0.2, seed=1234):
    """One LLM step: prompt -> text -> validated JSON record + narrative.

    Two things are enforced here and discussed in the report:
      * every numeric field is COMPUTED and injected, never taken from the model;
      * `quality_flag` is a safety gate derived from the pixels, so it is also
        computed - the model may phrase it but may not overrule it. Any
        disagreement is written to `corrections` rather than silently dropped.
    """
    prompt = prompt_template.replace('{measurements}', json.dumps(m, indent=2, default=float))
    text, source = call_llm(prompt, temperature=temperature, seed=seed)
    if text is None:
        text, source = stub_reply(m, kind), 'offline-stub'
    raw = extract_json(text)
    d, sh, q = classify(m)
    if kind == 't2':
        keys = ['n_objects', 'density_class', 'shape_regularity', 'quality_flag']
        computed = {'n_objects': int(m['n_objects']), 'quality_flag': q}
    else:
        keys = ['image_id', 'n_objects', 'mean_area', 'density_class', 'quality_flag']
        computed = {'image_id': m['image_id'], 'n_objects': int(m['n_objects']),
                    'mean_area': round(float(m['mean_area']), 1), 'quality_flag': q}
    rec, fixes = validate(raw, computed, keys)
    if raw and str(raw.get('density_class', '')).lower() != d:
        fixes.append(f"density_class: model said {raw.get('density_class')!r}, "
                     f"rule says {d!r} (model's label kept)")
    return dict(prompt=prompt, raw_text=text, record=rec, corrections=fixes,
                narrative=split_narrative(text), source=source,
                reference_labels=dict(density_class=d, shape_regularity=sh, quality_flag=q))

In [ ]:
t2 = run_step(PROMPT_T2, m_otsu, kind='t2')
print('source     :', t2['source'])
print('narrative  :', t2['narrative'])
print('record     :', json.dumps(t2['record'], indent=2))
print('corrections:', t2['corrections'])
print('ground truth n_objects:', int(meta.loc[meta.image_id==IID,'n_objects'].iloc[0]),
      '| density:', meta.loc[meta.image_id==IID,'density'].iloc[0])
(OUT/'task2_result.json').write_text(json.dumps({'measurements':m_otsu, **{k:t2[k] for k in
    ['prompt','narrative','record','corrections','source']}}, indent=2, default=float))

---
## Task 3 — U-Net segmentation (PyTorch)

### 3.1 Architecture and training setup

A deliberately small U-Net: 3 encoder levels with `DoubleConv` blocks (Conv3×3 → BatchNorm → ReLU,
twice), max-pool downsampling, nearest-neighbour upsampling with skip concatenation, and a 1×1
output convolution. Base width 8 → ~0.12 M parameters, which is appropriate for 80 training images.

**Design choices and what they trade off**

| Choice | Rationale | Trade-off |
|---|---|---|
| base width 8, depth 3 | 80 images; a full 31 M-parameter U-Net would memorise them | less capacity for fine boundary detail |
| random 128×128 crops | 4× cheaper per step and acts as strong spatial augmentation | train/test resolution mismatch in BatchNorm statistics |
| flips + 90° rotations | nuclei have no canonical orientation, so these are label-preserving | no intensity augmentation, so the model never sees contrast variation — this is exactly why it fails on the corrupted images later |
| nearest-neighbour upsample instead of transposed conv | removes checkerboard artefacts, fewer parameters | slightly blurrier boundaries |
| best-epoch checkpointing on val Dice | 80 images overfit quickly | val split is used for model selection, so val Dice is mildly optimistic |

### 3.0 Release GPU memory held by Ollama

In [ ]:
# Free the GPU before training: unload the Ollama models so PyTorch gets the VRAM.
# They reload on demand when Task 4 needs them again.
for _m in [VISION_MODEL, TEXT_MODEL]:
    if _m:
        try:
            urllib.request.urlopen(urllib.request.Request(
                OLLAMA_URL, data=json.dumps({'model': _m, 'keep_alive': 0}).encode(),
                headers={'Content-Type': 'application/json'}), timeout=60)
        except Exception:
            pass
print('Ollama models unloaded')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
print('device:', DEVICE)

class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class SmallUNet(nn.Module):
    def __init__(self, base=8, depth=3, in_ch=1):
        super().__init__()
        chs = [base*2**i for i in range(depth+1)]
        self.depth = depth; self.chs = chs
        self.enc = nn.ModuleList()
        c = in_ch
        for ch in chs[:-1]:
            self.enc.append(DoubleConv(c, ch)); c = ch
        self.bott = DoubleConv(c, chs[-1])
        self.dec = nn.ModuleList([DoubleConv(chs[i+1]+chs[i], chs[i]) for i in range(depth)])
        self.out = nn.Conv2d(chs[0], 1, 1)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        skips = []
        for e in self.enc:
            x = e(x); skips.append(x); x = self.pool(x)
        x = self.bott(x)
        for i in range(self.depth-1, -1, -1):
            x = F.interpolate(x, scale_factor=2, mode='nearest')
            x = torch.cat([x, skips[i]], dim=1)
            x = self.dec[i](x)
        return self.out(x)

net = SmallUNet()
print('parameters:', sum(p.numel() for p in net.parameters()))

In [ ]:
class NucleiDS(Dataset):
    """Random 128x128 crops + flips/rotations for training; full 256x256 images for eval."""
    def __init__(self, X, Y, crop=None, augment=False, seed=0):
        self.X, self.Y, self.crop, self.aug = X, Y, crop, augment
        self.rng = np.random.default_rng(seed)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x, y = self.X[i], self.Y[i]
        if self.crop:
            r0 = int(self.rng.integers(0, x.shape[0]-self.crop+1))
            c0 = int(self.rng.integers(0, x.shape[1]-self.crop+1))
            x = x[r0:r0+self.crop, c0:c0+self.crop]; y = y[r0:r0+self.crop, c0:c0+self.crop]
        if self.aug:
            k = int(self.rng.integers(0,4)); x = np.rot90(x,k); y = np.rot90(y,k)
            if self.rng.random() < .5: x = x[:, ::-1]; y = y[:, ::-1]
            if self.rng.random() < .5: x = x[::-1]; y = y[::-1]
        return (torch.from_numpy(np.ascontiguousarray(x))[None].float(),
                torch.from_numpy(np.ascontiguousarray(y))[None].float())

tr_ids, Xtr, Ytr = load_split('train')
va_ids, Xva, Yva = load_split('val')
te_ids, Xte, Yte = load_split('test')
print(Xtr.shape, Xva.shape, Xte.shape)

train_dl = DataLoader(NucleiDS(Xtr, Ytr, crop=CROP, augment=True, seed=1), batch_size=8, shuffle=True)
val_dl   = DataLoader(NucleiDS(Xva, Yva), batch_size=4)

### 3.2 Losses

With ~8% foreground, plain BCE is dominated by easy background pixels; soft Dice optimises overlap
directly but has an unstable gradient when the prediction is near-empty. The sum of the two is the
standard compromise, and the ablation below tests whether it actually helps here.

In [ ]:
def dice_loss(logits, y, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p*y).sum()
    return 1 - (2*inter + eps)/(p.sum() + y.sum() + eps)

bce_fn = nn.BCEWithLogitsLoss()

def make_loss(kind):
    if kind == 'bce':      return lambda lg,y: bce_fn(lg,y)
    if kind == 'dice':     return lambda lg,y: dice_loss(lg,y)
    return lambda lg,y: bce_fn(lg,y) + dice_loss(lg,y)

@torch.no_grad()
def dice_iou_np(pred, gt, eps=1e-7):
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = np.logical_and(pred,gt).sum()
    return ((2*inter+eps)/(pred.sum()+gt.sum()+eps),
            (inter+eps)/(np.logical_or(pred,gt).sum()+eps))

@torch.no_grad()
def evaluate(model, X, Y, thr=0.5, bs=4):
    model.eval()
    probs = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i+bs])[:,None].float().to(DEVICE)
        probs.append(torch.sigmoid(model(xb)).cpu().numpy())
    probs = np.concatenate(probs)[:,0]
    ds, ious = zip(*[dice_iou_np(probs[k]>thr, Y[k]>0.5) for k in range(len(X))])
    return np.array(ds), np.array(ious), probs

In [ ]:
def train_unet(loss_kind='bce_dice', epochs=30, lr=1e-3, seed=SEED, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    model = SmallUNet().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = make_loss(loss_kind)
    hist, best = [], (-1, None, -1)
    for ep in range(1, epochs+1):
        model.train(); tot = nb = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            lg = model(xb); loss = crit(lg, yb)
            loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        vd, vi, _ = evaluate(model, Xva, Yva)
        model.eval()
        with torch.no_grad():
            vl = np.mean([crit(model(torch.from_numpy(Xva[i:i+4])[:,None].float().to(DEVICE)),
                               torch.from_numpy(Yva[i:i+4])[:,None].float().to(DEVICE)).item()
                          for i in range(0,len(Xva),4)])
        hist.append(dict(epoch=ep, train_loss=tot/nb, val_loss=vl,
                         val_dice=vd.mean(), val_iou=vi.mean()))
        if vd.mean() > best[0]:
            best = (vd.mean(), {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}, ep)
        if verbose:
            print(f'[{loss_kind}] ep{ep:02d} train {tot/nb:.4f} val {vl:.4f} '
                  f'dice {vd.mean():.4f} iou {vi.mean():.4f}')
    model.load_state_dict(best[1])
    return model, pd.DataFrame(hist), best[2]

### 3.3 Loss ablation (extension) and final model

In [ ]:
ablation, models = {}, {}
for kind in ['bce','dice','bce_dice']:
    t0 = time.time()
    m, h, bep = train_unet(kind, epochs=30, verbose=False)
    vd, vi, _ = evaluate(m, Xva, Yva)
    h.to_csv(OUT/f'history_{kind}.csv', index=False)
    ablation[kind] = dict(val_dice=float(vd.mean()), val_iou=float(vi.mean()),
                          val_dice_std=float(vd.std()), val_dice_min=float(vd.min()),
                          best_epoch=int(bep), minutes=round((time.time()-t0)/60,2))
    models[kind] = m
    print(kind, ablation[kind])

abl = pd.DataFrame(ablation).T
display(abl.round(4))
best_kind = abl.val_dice.idxmax(); unet = models[best_kind]
print('best loss:', best_kind)
torch.save(unet.state_dict(), OUT/'unet_best.pt')

### 3.4 Loss and Dice curves

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(13,3.4))
cols = {'bce':'#3b6ea5','dice':'#c1443c','bce_dice':'#4c9f70'}
for kind,c in cols.items():
    h = pd.read_csv(OUT/f'history_{kind}.csv')
    ax[0].plot(h.epoch, h.train_loss, color=c, label=kind)
    ax[1].plot(h.epoch, h.val_loss,  color=c, label=kind)
    ax[2].plot(h.epoch, h.val_dice,  color=c, label=kind)
ax[0].set_title('training loss'); ax[1].set_title('validation loss'); ax[2].set_title('validation Dice')
for a in ax: a.set_xlabel('epoch'); a.legend(fontsize=8); a.grid(alpha=.3)
ax[2].axhline(0.9742, ls='--', c='k', lw=1)
ax[2].text(1, 0.945, 'Otsu baseline', fontsize=8)
plt.tight_layout(); plt.savefig(FIGS/'fig4_curves.png', dpi=160, bbox_inches='tight'); plt.show()

### 3.5 Validation results: input / ground truth / prediction

In [ ]:
vd, vi, vprob = evaluate(unet, Xva, Yva)
val_df = pd.DataFrame(dict(image_id=va_ids, dice=vd, iou=vi)).merge(
            meta[['image_id','density','n_objects']], on='image_id')
val_df.to_csv(OUT/'unet_val_per_image.csv', index=False)
print(f'VALIDATION  mean Dice {vd.mean():.4f} +/- {vd.std():.4f}   mean IoU {vi.mean():.4f}')
display(val_df.groupby('density')[['dice','iou']].agg(['mean','min']).round(4))
display(val_df.sort_values('dice').head(3).round(4))

# Otsu baseline on the same split, for the comparison in the report
otsu_rows = []
for split in ['val','test']:
    for iid in split_ids(split):
        g = load_gray(DATA_ROOT/split/'images'/f'{iid}.png')
        mk = load_mask(DATA_ROOT/split/'masks'/f'{iid}.png')
        b,_ = otsu_segment(g); d,i_ = dice_iou_np(b, mk)
        otsu_rows.append(dict(image_id=iid, split=split, dice=d, iou=i_))
otsu_df = pd.DataFrame(otsu_rows); otsu_df.to_csv(OUT/'otsu_baseline.csv', index=False)
print(otsu_df.groupby('split')[['dice','iou']].mean().round(4))

In [ ]:
show = list(val_df.sort_values('dice').image_id[:2]) + [val_df.sort_values('dice').image_id.iloc[-1]]
fig, ax = plt.subplots(len(show), 4, figsize=(11.5, 2.9*len(show)))
for r, iid in enumerate(show):
    k = va_ids.index(iid)
    g, gt, pr = Xva[k], Yva[k], vprob[k] > 0.5
    ot,_ = otsu_segment(g)
    d_u,_ = dice_iou_np(pr, gt); d_o,_ = dice_iou_np(ot, gt)
    for c,(img,t) in enumerate(zip([g, gt, pr, ot],
            ['input', 'ground truth', f'U-Net (Dice {d_u:.3f})', f'Otsu (Dice {d_o:.3f})'])):
        ax[r,c].imshow(img, cmap='gray', vmin=0, vmax=(np.percentile(g,99.9) if c==0 else 1))
        ax[r,c].set_title(t, fontsize=9); ax[r,c].set_xticks([]); ax[r,c].set_yticks([])
    ax[r,0].set_ylabel(iid, fontsize=9)
plt.tight_layout(); plt.savefig(FIGS/'fig5_unet_panels.png', dpi=160, bbox_inches='tight'); plt.show()

---
## Task 4 — Hybrid pipeline on the unseen test images

`U-Net mask → regionprops feature table → validated JSON record → narrative`, aggregated to a CSV.

In [ ]:
records, narratives, measurements = [], {}, []
for iid in te_ids:
    k = te_ids.index(iid)
    g, gt = Xte[k], Yte[k]
    with torch.no_grad():
        prob = torch.sigmoid(unet(torch.from_numpy(g)[None,None].float().to(DEVICE)))[0,0].cpu().numpy()
    pred = prob > 0.5
    d, i_ = dice_iou_np(pred, gt)
    m, feat_df = measure_field(iid, g, pred)
    res = run_step(PROMPT_T4, m, kind='t4')
    rec = dict(res['record'])
    rec.update(dice_vs_gt=round(d,4), iou_vs_gt=round(i_,4),
               n_objects_true=int(meta.loc[meta.image_id==iid,'n_objects'].iloc[0]),
               true_density=str(meta.loc[meta.image_id==iid,'density'].iloc[0]),
               area_fraction=m['area_fraction'], mean_solidity=m['mean_solidity'],
               mean_eccentricity=m['mean_eccentricity'],
               contrast_score=m['contrast_score'], focus_score=m['focus_score'],
               n_corrections=len(res['corrections']), llm_source=res['source'])
    records.append(rec); narratives[iid] = res['narrative']; measurements.append(m)

pipeline_df = pd.DataFrame(records)
pipeline_df.to_csv(OUT/'pipeline_records.csv', index=False)
(OUT/'pipeline_narratives.json').write_text(json.dumps(narratives, indent=2))
display(pipeline_df)
print('CSV written to', OUT/'pipeline_records.csv')

In [ ]:
print('mean test Dice :', round(pipeline_df.dice_vs_gt.mean(),4))
print('mean test IoU  :', round(pipeline_df.iou_vs_gt.mean(),4))
print('count MAE      :', round((pipeline_df.n_objects-pipeline_df.n_objects_true).abs().mean(),2))
print('count bias     :', round((pipeline_df.n_objects-pipeline_df.n_objects_true).mean(),2))
print('quality flags  :', pipeline_df.quality_flag.value_counts().to_dict())
print('total validator corrections:', int(pipeline_df.n_corrections.sum()))
print()
eg = te_ids[0]
print('EXAMPLE RECORD\n', json.dumps(records[0], indent=2))
print('\nEXAMPLE NARRATIVE\n', narratives[eg])

---
## Extension — Robustness: tracing a corruption through the pipeline

`test_000` is run three ways: clean, heavily blurred, and low contrast. At each stage we ask whether
the corruption is *detectable* — not merely present.

In [ ]:
base = 'test_000'
variants = {'clean':       DATA_ROOT/'test'/'images'/f'{base}.png',
            'blur':        DATA_ROOT/'test_corrupted'/'images'/f'{base}_blur.png',
            'lowcontrast': DATA_ROOT/'test_corrupted'/'images'/f'{base}_lowcontrast.png'}
gt = load_mask(DATA_ROOT/'test'/'masks'/f'{base}.png')

rows, panels = [], {}
for name, path in variants.items():
    g = load_gray(path)
    ot,_ = otsu_segment(g)
    with torch.no_grad():
        prob = torch.sigmoid(unet(torch.from_numpy(g)[None,None].float().to(DEVICE)))[0,0].cpu().numpy()
    pred = prob > 0.5
    m,_  = measure_field(f'{base}_{name}', g, pred)
    mo,_ = measure_field(f'{base}_{name}', g, ot)
    res = run_step(PROMPT_T4, m, kind='t4')
    d_u,_ = dice_iou_np(pred, gt); d_o,_ = dice_iou_np(ot, gt)
    rows.append(dict(variant=name, **quality_metrics(g), unet_dice=round(d_u,4),
                     otsu_dice=round(d_o,4), unet_n=m['n_objects'], otsu_n=mo['n_objects'],
                     unet_mean_area=m['mean_area'], mean_solidity=m['mean_solidity'],
                     density_class=res['record']['density_class'],
                     quality_flag=res['record']['quality_flag']))
    narratives[f'{base}_{name}'] = res['narrative']
    panels[name] = (g, ot, pred)

rob = pd.DataFrame(rows); rob.to_csv(OUT/'robustness.csv', index=False)
display(rob)
for k in [f'{base}_clean', f'{base}_blur', f'{base}_lowcontrast']:
    print(f'\n[{k}]\n{narratives[k]}')

In [ ]:
fig, ax = plt.subplots(3, 4, figsize=(11.5, 8.4))
for r,(name,(g,ot,pr)) in enumerate(panels.items()):
    rr = rob[rob.variant==name].iloc[0]
    for c,(img,t,xl) in enumerate(zip([g,ot,pr,gt],
            ['input','Otsu mask','U-Net mask','ground truth'],
            ['', f'Dice {rr.otsu_dice:.3f}', f'Dice {rr.unet_dice:.3f}', ''])):
        ax[r,c].imshow(img, cmap='gray', vmin=0, vmax=(float(np.percentile(g,99.9)) if c==0 else 1))
        if r==0: ax[r,c].set_title(t, fontsize=9)
        ax[r,c].set_xlabel(xl, fontsize=8); ax[r,c].set_xticks([]); ax[r,c].set_yticks([])
    ax[r,0].set_ylabel(name, fontsize=11)
plt.tight_layout(); plt.savefig(FIGS/'fig6_robustness.png', dpi=155, bbox_inches='tight'); plt.show()

**Where the corruption first becomes detectable.** The image-level quality statistics fire before
anything downstream: `focus_score` collapses by roughly two orders of magnitude under blur and
`contrast_score` falls to about an eighth of its clean value under contrast loss, while the mask,
the feature table and the narrative are all still superficially well-formed. That is the argument
for computing the gate from the pixels rather than asking the LLM to notice — the narrative sounds
exactly as confident on the corrupted image as on the clean one, and only the flag distinguishes them.

---
## Summary of every number quoted in the report

In [ ]:
summary = dict(
  dataset=dict(n_images=int(len(meta)), splits=meta.split.value_counts().to_dict(),
               n_objects_range=[int(meta.n_objects.min()), int(meta.n_objects.max())],
               fg_pixel_fraction=round(float(len(fg)/(len(fg)+len(bg))),4)),
  otsu=dict(val_dice=round(float(otsu_df[otsu_df.split=='val'].dice.mean()),4),
            val_iou=round(float(otsu_df[otsu_df.split=='val'].iou.mean()),4),
            test_dice=round(float(otsu_df[otsu_df.split=='test'].dice.mean()),4)),
  unet=dict(params=int(sum(p.numel() for p in unet.parameters())), best_loss=best_kind,
            val_dice=round(float(vd.mean()),4), val_iou=round(float(vi.mean()),4),
            val_dice_std=round(float(vd.std()),4),
            test_dice=round(float(pipeline_df.dice_vs_gt.mean()),4),
            test_iou=round(float(pipeline_df.iou_vs_gt.mean()),4)),
  ablation=ablation,
  pipeline=dict(count_mae=round(float((pipeline_df.n_objects-pipeline_df.n_objects_true).abs().mean()),2),
                count_bias=round(float((pipeline_df.n_objects-pipeline_df.n_objects_true).mean()),2),
                quality_flags=pipeline_df.quality_flag.value_counts().to_dict(),
                corrections=int(pipeline_df.n_corrections.sum())),
  robustness=rob.to_dict('records'),
  llm_source=('ollama' if USE_OLLAMA else 'offline-stub'))
(OUT/'report_numbers.json').write_text(json.dumps(summary, indent=2, default=float))
print(json.dumps(summary, indent=2, default=float))

---
### Files produced

| Path | Contents |
|---|---|
| `figs/fig1_samples.png` | sample images and ground-truth masks by density regime |
| `figs/fig2_eda_hist.png` | intensity histograms and density-regime scatter |
| `figs/fig3_classical.png` | Otsu pipeline: histogram, mask, components, watershed |
| `figs/fig4_curves.png` | training/validation loss and validation Dice for all three losses |
| `figs/fig5_unet_panels.png` | input / ground truth / U-Net / Otsu for validation images |
| `figs/fig6_robustness.png` | clean vs blurred vs low-contrast propagation |
| `out/pipeline_records.csv` | **Task 4 deliverable** — aggregated JSON records for all test images |
| `out/report_numbers.json` | every number quoted in the report |

### Final self-check — run this before submitting

In [ ]:
ok = True
print('vision model      :', VISION_MODEL or 'NONE (Task 1 skipped)')
print('LLM sources       :', sorted(pipeline_df.llm_source.unique()))
print('validator fixes   :', int(pipeline_df.n_corrections.sum()))
blank = sum(1 for v in narratives.values() if not str(v).strip())
print('blank narratives  :', blank, 'of', len(narratives))
for cond, msg in [(VISION_MODEL is not None, 'Task 1 produced no VLM output'),
                  (set(pipeline_df.llm_source) == {'ollama'}, 'some records used the offline stub'),
                  (blank == 0, 'some narratives are empty')]:
    if not cond:
        ok = False; print('  PROBLEM:', msg)
print('\nALL CHECKS PASSED' if ok else '\nfix the problems above, then re-run')